# Flight School Demo

This notebook demonstrates the Bluebird Gymnasium `FlightSchoolEnv-v0` environment. Flight School uses the X-plus sector with an infinite traffic generator, so aircraft continue to enter the sector during the episode.

## Imports

In [ ]:
from enum import Enum
from pprint import pprint

import gymnasium as gym
import IPython.display
import matplotlib.pyplot as plt
import numpy as np

import bluebird_gymnasium
from bluebird_gymnasium.envs import BaseEnv, FlightSchoolEnv

## Helpers

In [ ]:
def actions_to_enum(env: BaseEnv) -> Enum:
    actions_map = env.get_action_parser().action_formatter_map
    enum_items = {}

    for action_int, action_name in actions_map.items():
        parts = action_name.split("__")
        base_name = parts[0]
        magnitude = parts[1] if len(parts) == 2 else None

        if base_name == "action_noop":
            enum_name = "NOOP"
        elif base_name == "simple_heading_left":
            enum_name = "LEFT"
        elif base_name == "simple_heading_right":
            enum_name = "RIGHT"
        elif base_name == "simple_heading_route_parallel":
            enum_name = "ROUTE_PARALLEL"
        else:
            enum_name = base_name.upper()

        if magnitude is not None:
            enum_name = f"{enum_name}_{magnitude}"

        enum_items[enum_name] = action_int

    return Enum("Actions", list(enum_items.items()))


def active_aircraft_count(env: BaseEnv) -> int:
    return len(env.get_manager().environment.aircraft)

## Scenario Config

The default Flight School config uses the X-plus sector, two starter aircraft, and a gradual increasing spawn rate. This notebook runs a 10-minute Gymnasium episode from that generator.

In [ ]:
seed = 7

config = FlightSchoolEnv.get_default_env_config()
config.scenario_config["args"]["random_seed"] = seed
config.scenario_duration = 10 * 60
config.view_config["type"] = "centralized"

pprint(config.scenario_config)

## Instantiate The Environment

In [ ]:
env = gym.make("FlightSchoolEnv-v0", config=config).unwrapped
env.set_render_mode("human")

obs, info = env.reset(seed=seed)
Actions = actions_to_enum(env)

print(f"Observation shape: {obs.shape}")
print(f"Action count: {env.action_space.n}")
print(f"Active aircraft: {active_aircraft_count(env)}")
print(Actions.__members__)

## Baseline Run

This baseline uses `NOOP` at each step. It is useful as a smoke test and as a reference trace before adding an agent.

In [ ]:
num_steps = config.scenario_duration // config.scenario_sec_per_step
noop = Actions.NOOP.value

reward_history = []
traffic_history = []
time_history = []

obs, info = env.reset(seed=seed)

for step in range(num_steps):
    obs, reward, done, truncated, info = env.step(noop)
    reward_history.append(reward)
    traffic_history.append(active_aircraft_count(env))
    time_history.append(env.get_manager().environment.time)

    if step % 10 == 0 or done or truncated:
        env.render()
        IPython.display.display(env.radar.figure)
        IPython.display.clear_output(wait=True)

    if done or truncated:
        break

env.render()
IPython.display.display(env.radar.figure)
print(f"Completed {len(reward_history)} steps")

## Reward And Traffic Trace

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(time_history, reward_history, marker="o", linewidth=1)
axes[0].set_ylabel("Reward")
axes[0].grid(True, alpha=0.3)

axes[1].step(time_history, traffic_history, where="post")
axes[1].set_xlabel("Simulation time (s)")
axes[1].set_ylabel("Active aircraft")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

## Appendix: Direct Class Construction

`gym.make` is convenient when using Gymnasium wrappers. Direct construction is useful when you want the concrete Bluebird environment instance immediately.

In [ ]:
direct_env = FlightSchoolEnv(config=config)
direct_obs, direct_info = direct_env.reset(seed=seed)

print(type(direct_env).__name__)
print(direct_obs.shape)
print(type(direct_env.scenario_manager).__name__)